In [3]:
import pandas as pd
import numpy as np

In [4]:

df = pd.read_csv("synthetic_raw_data.csv")

In [5]:
df.shape

(50000, 16)

In [6]:
df.columns

Index(['timestamp', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'geo_country',
       'asn', 'txid', 'input_addresses', 'output_addresses', 'input_amounts',
       'output_amounts', 'fee', 'script_type', 'is_anomaly', 'pattern_type'],
      dtype='str')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   timestamp         50000 non-null  int64  
 1   src_ip            48961 non-null  str    
 2   src_port          48961 non-null  float64
 3   dst_ip            48961 non-null  str    
 4   dst_port          50000 non-null  int64  
 5   geo_country       48961 non-null  str    
 6   asn               48961 non-null  str    
 7   txid              50000 non-null  str    
 8   input_addresses   49494 non-null  str    
 9   output_addresses  50000 non-null  str    
 10  input_amounts     49494 non-null  str    
 11  output_amounts    50000 non-null  str    
 12  fee               50000 non-null  float64
 13  script_type       50000 non-null  str    
 14  is_anomaly        50000 non-null  int64  
 15  pattern_type      50000 non-null  str    
dtypes: float64(2), int64(3), str(11)
memory usage: 17.3

In [8]:
df.isnull().sum()

timestamp              0
src_ip              1039
src_port            1039
dst_ip              1039
dst_port               0
geo_country         1039
asn                 1039
txid                   0
input_addresses      506
output_addresses       0
input_amounts        506
output_amounts         0
fee                    0
script_type            0
is_anomaly             0
pattern_type           0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df_clean = df.copy()

In [11]:
text_columns = df_clean.select_dtypes(include="str").columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

In [12]:
" US"
"US "

'US '

In [13]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates()

after = len(df_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 50000
Rows after: 50000
Duplicates removed: 0


In [14]:
df_clean["txid"].duplicated().sum()

np.int64(0)

In [15]:
missing = df_clean.isnull().sum()

missing[missing > 0]

src_ip             1039
src_port           1039
dst_ip             1039
geo_country        1039
asn                1039
input_addresses     506
input_amounts       506
dtype: int64

In [16]:
missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100

missing_report = pd.DataFrame({
    "missing_count": df_clean.isnull().sum(),
    "missing_percentage": missing_percentage
})

missing_report[missing_report["missing_count"] > 0]

,missing_count,missing_percentage
src_ip,1039,2.078
src_port,1039,2.078
dst_ip,1039,2.078
geo_country,1039,2.078
asn,1039,2.078
input_addresses,506,1.012
input_amounts,506,1.012


In [17]:
for col in ["geo_country", "script_type", "pattern_type", "is_anomaly"]:
    print("\n---", col, "---")
    print(df_clean[col].value_counts(dropna=False))


--- geo_country ---
geo_country
JP     16303
DE     15974
US     15257
NaN     1039
RU       508
NL       460
SC       459
Name: count, dtype: int64

--- script_type ---
script_type
v0_p2wpkh    12880
p2pkh        12736
p2sh         12366
v1_p2tr      12018
Name: count, dtype: int64

--- pattern_type ---
pattern_type
normal                   48545
coinjoin_mixer             380
fast_flux_structuring      359
ransom_fanout              358
peeling_chain              358
Name: count, dtype: int64

--- is_anomaly ---
is_anomaly
0    48545
1     1455
Name: count, dtype: int64


In [18]:
invalid_src_ports = df_clean[
    df_clean["src_port"].notna() &
    ~df_clean["src_port"].between(0, 65535)
]

invalid_dst_ports = df_clean[
    ~df_clean["dst_port"].between(0, 65535)
]

print("Invalid source ports:", len(invalid_src_ports))
print("Invalid destination ports:", len(invalid_dst_ports))

Invalid source ports: 0
Invalid destination ports: 0


In [19]:
import ipaddress

def check_ip(ip):
    if pd.isna(ip):
        return True
    try:
        ipaddress.ip_address(ip)
        return True
    except ValueError:
        return False

print("Invalid source IPs:",
      (~df_clean["src_ip"].apply(check_ip)).sum())

print("Invalid destination IPs:",
      (~df_clean["dst_ip"].apply(check_ip)).sum())

Invalid source IPs: 0
Invalid destination IPs: 0


In [20]:
print("Negative fees:", (df_clean["fee"] < 0).sum())

Negative fees: 0


In [21]:
def count_items(value):
    if pd.isna(value):
        return 0
    return len(str(value).split(";"))

input_mismatch = (
    df_clean["input_addresses"].apply(count_items)
    != df_clean["input_amounts"].apply(count_items)
)

output_mismatch = (
    df_clean["output_addresses"].apply(count_items)
    != df_clean["output_amounts"].apply(count_items)
)

print("Input mismatch:", input_mismatch.sum())
print("Output mismatch:", output_mismatch.sum())

Input mismatch: 0
Output mismatch: 0


In [22]:
df_clean["network_data_missing"] = (
    df_clean[["src_ip", "src_port", "dst_ip", "geo_country", "asn"]]
    .isnull()
    .any(axis=1)
)

df_clean["input_data_missing"] = (
    df_clean[["input_addresses", "input_amounts"]]
    .isnull()
    .any(axis=1)
)

In [23]:
print("Rows with missing network data:",
      df_clean["network_data_missing"].sum())

print("Rows with missing input data:",
      df_clean["input_data_missing"].sum())

Rows with missing network data: 1039
Rows with missing input data: 506


In [24]:
df_clean.dtypes

timestamp                 int64
src_ip                      str
src_port                float64
dst_ip                      str
dst_port                  int64
geo_country                 str
asn                         str
txid                        str
input_addresses             str
output_addresses            str
input_amounts               str
output_amounts              str
fee                     float64
script_type                 str
is_anomaly                int64
pattern_type                str
network_data_missing       bool
input_data_missing         bool
dtype: object

In [25]:
print("Total rows:", len(df_clean))
print("Total columns:", len(df_clean.columns))
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate TXIDs:", df_clean["txid"].duplicated().sum())

Total rows: 50000
Total columns: 18
Duplicate rows: 0
Duplicate TXIDs: 0


In [26]:
final_missing = pd.DataFrame({
    "missing_count": df_clean.isnull().sum(),
    "missing_percentage": (df_clean.isnull().sum() / len(df_clean)) * 100
})

final_missing[final_missing["missing_count"] > 0]

,missing_count,missing_percentage
src_ip,1039,2.078
src_port,1039,2.078
dst_ip,1039,2.078
geo_country,1039,2.078
asn,1039,2.078
input_addresses,506,1.012
input_amounts,506,1.012


In [27]:
df_clean.to_csv("synthetic_cleaned_data.csv", index=False)

In [28]:
import os

print(os.path.exists("synthetic_cleaned_data.csv"))

True


In [29]:
df

,timestamp,src_ip,src_port,dst_ip,dst_port,geo_country,asn,txid,input_addresses,output_addresses,input_amounts,output_amounts,fee,script_type,is_anomaly,pattern_type
0,1785524364,191.38.23.68,1924.0,141.12.143.177,8333,US,AS15169,0x8e2ea753485c4d6280c6162751e4128f43d15bceab12...,bc1qd5ceb396aa905cf7413b8ca35718,bc1q357c85c3c26dbc5d499b784c873a;bc1pd6054fe66...,1.1649,0.5821;0.5821,0.00077,p2sh,0,normal
1,1785524953,210.68.57.187,41855.0,48.238.118.132,8333,US,AS15169,0x7855e43fcb8634e5a45526001edd480c32fd0fba778b...,bc1q163eb005199626c8144b4bc12f40,1878492084022132c52fc352bf2e4;bc1qfb13c765e04e...,0.157,0.0784;0.0784,0.00025,v0_p2wpkh,0,normal
2,1785525568,201.212.50.173,31495.0,85.138.195.222,8333,DE,AS3320,0x5e6d0069dc8b60c0943a146b544d3e92e65787438ae3...,16a0510ca1a7f26978b421a99d43c;bc1q45947e2b9ebf...,1f6cc470ba322fca5de8f4d35360f;bc1qc06cff562df9...,0.678;0.678,0.6775;0.6775,0.00098,p2pkh,0,normal
3,1785526850,206.137.211.130,13241.0,212.121.13.166,8333,JP,AS2516,0x01df5f1b25cc0e9af79c33d746a99fcb45a0e2e8174f...,bc1q93722a50b2fe81788ac3035f86f0;1222043552f51...,1bd82a121b93ad54fcbba60309a67,0.5353;0.5353,1.0702,0.00054,p2sh,0,normal
4,1785526859,191.56.238.116,443.0,212.34.74.15,8333,SC,AS9009,0xf72309ed760e7ecfb2e6106589fa4e2a51d817b68546...,bc1qfc8cc112c8dfbfa32a8d517852c0,16785556c2747f7a7881e280998d5;176931d14707a92c...,51.5352,2.1734;2.1734;2.1734;2.1734;2.1734;2.1734;2.17...,1.54610,p2pkh,1,ransom_fanout
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,1830575345,149.167.206.172,1727.0,203.229.188.194,8333,JP,AS2516,0x8353f4445f1fb4382a2d687372d1c7109e4824b8bf14...,1a1046f19911509f493a6fed026f6;306da954363ae135...,104d6606438f46d9aaaef409729a6;31efcbcfd227a9b6...,0.4341;0.4341,0.4339;0.4339,0.00044,p2pkh,0,normal
49996,1830576322,NaN,NaN,NaN,8333,NaN,NaN,0x55b1a4b3d5fe1f7f0ccffef1015cc094fc60d71dd33f...,bc1q9ca6fd49deac66dd2ff3f1e256e6,bc1p7e964c81ae367e820e760ab04f5bf72b;bc1q4336d...,0.5647,0.282;0.282,0.00061,v1_p2tr,0,normal
49997,1830577572,193.199.63.136,38618.0,207.11.189.59,8333,US,AS15169,0xd1b513ea3a41d067d2c717e88d4b66a249da243b2c30...,1bf315632f584a38fe9f65edc8000,bc1qfcb0038c5fab2e2c0b640c1f553d,2.097,2.0965,0.00053,v0_p2wpkh,0,normal
49998,1830578864,221.108.180.212,8732.0,208.43.4.140,8333,DE,AS3320,0xbaaf9f9cff618d3503ecc0abcada9604fba01e3d4a01...,1872f2cbf06bfa1c63181b6a95669;bc1q63c29b01d234...,100d1c15f4720086b57bbd628a176;39d6a4c140b5590e...,0.7385;0.7385,0.7382;0.7382,0.00060,p2pkh,0,normal


In [29]:
df

,timestamp,src_ip,src_port,dst_ip,dst_port,geo_country,asn,txid,input_addresses,output_addresses,input_amounts,output_amounts,fee,script_type,is_anomaly,pattern_type
0,1785524364,191.38.23.68,1924.0,141.12.143.177,8333,US,AS15169,0x8e2ea753485c4d6280c6162751e4128f43d15bceab12...,bc1qd5ceb396aa905cf7413b8ca35718,bc1q357c85c3c26dbc5d499b784c873a;bc1pd6054fe66...,1.1649,0.5821;0.5821,0.00077,p2sh,0,normal
1,1785524953,210.68.57.187,41855.0,48.238.118.132,8333,US,AS15169,0x7855e43fcb8634e5a45526001edd480c32fd0fba778b...,bc1q163eb005199626c8144b4bc12f40,1878492084022132c52fc352bf2e4;bc1qfb13c765e04e...,0.157,0.0784;0.0784,0.00025,v0_p2wpkh,0,normal
2,1785525568,201.212.50.173,31495.0,85.138.195.222,8333,DE,AS3320,0x5e6d0069dc8b60c0943a146b544d3e92e65787438ae3...,16a0510ca1a7f26978b421a99d43c;bc1q45947e2b9ebf...,1f6cc470ba322fca5de8f4d35360f;bc1qc06cff562df9...,0.678;0.678,0.6775;0.6775,0.00098,p2pkh,0,normal
3,1785526850,206.137.211.130,13241.0,212.121.13.166,8333,JP,AS2516,0x01df5f1b25cc0e9af79c33d746a99fcb45a0e2e8174f...,bc1q93722a50b2fe81788ac3035f86f0;1222043552f51...,1bd82a121b93ad54fcbba60309a67,0.5353;0.5353,1.0702,0.00054,p2sh,0,normal
4,1785526859,191.56.238.116,443.0,212.34.74.15,8333,SC,AS9009,0xf72309ed760e7ecfb2e6106589fa4e2a51d817b68546...,bc1qfc8cc112c8dfbfa32a8d517852c0,16785556c2747f7a7881e280998d5;176931d14707a92c...,51.5352,2.1734;2.1734;2.1734;2.1734;2.1734;2.1734;2.17...,1.54610,p2pkh,1,ransom_fanout
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,1830575345,149.167.206.172,1727.0,203.229.188.194,8333,JP,AS2516,0x8353f4445f1fb4382a2d687372d1c7109e4824b8bf14...,1a1046f19911509f493a6fed026f6;306da954363ae135...,104d6606438f46d9aaaef409729a6;31efcbcfd227a9b6...,0.4341;0.4341,0.4339;0.4339,0.00044,p2pkh,0,normal
49996,1830576322,NaN,NaN,NaN,8333,NaN,NaN,0x55b1a4b3d5fe1f7f0ccffef1015cc094fc60d71dd33f...,bc1q9ca6fd49deac66dd2ff3f1e256e6,bc1p7e964c81ae367e820e760ab04f5bf72b;bc1q4336d...,0.5647,0.282;0.282,0.00061,v1_p2tr,0,normal
49997,1830577572,193.199.63.136,38618.0,207.11.189.59,8333,US,AS15169,0xd1b513ea3a41d067d2c717e88d4b66a249da243b2c30...,1bf315632f584a38fe9f65edc8000,bc1qfcb0038c5fab2e2c0b640c1f553d,2.097,2.0965,0.00053,v0_p2wpkh,0,normal
49998,1830578864,221.108.180.212,8732.0,208.43.4.140,8333,DE,AS3320,0xbaaf9f9cff618d3503ecc0abcada9604fba01e3d4a01...,1872f2cbf06bfa1c63181b6a95669;bc1q63c29b01d234...,100d1c15f4720086b57bbd628a176;39d6a4c140b5590e...,0.7385;0.7385,0.7382;0.7382,0.00060,p2pkh,0,normal


In [30]:
def parse_amounts(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    try:
        return [float(x.strip()) for x in str(val).split(";") if x.strip() != ""]
    except ValueError:
        return np.nan

df_clean["input_amounts_parsed"] = df_clean["input_amounts"].apply(parse_amounts)
df_clean["output_amounts_parsed"] = df_clean["output_amounts"].apply(parse_amounts)

def validate_and_clean_addresses(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    cleaned_list = [str(x).strip() for x in str(val).split(";") if str(x).strip() != ""]
    return cleaned_list if len(cleaned_list) > 0 else np.nan

df_clean["input_addresses_parsed"] = df_clean["input_addresses"].apply(validate_and_clean_addresses)
df_clean["output_addresses_parsed"] = df_clean["output_addresses"].apply(validate_and_clean_addresses)

In [31]:
df_clean["geo_country"] = df_clean["geo_country"].fillna("Unknown")
df_clean["asn"] = df_clean["asn"].fillna("Unknown")

df_clean["timestamp"] = pd.to_numeric(df_clean["timestamp"], errors='coerce')
assert df_clean["timestamp"].notna().all()

categorical_cols = ["script_type", "geo_country", "asn"]
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str)

TARGET_COLUMN = "is_anomaly"
EVALUATION_COLUMN = "pattern_type"
IDENTIFIER_COLUMN = "txid"

df_clean["fee"] = pd.to_numeric(df_clean["fee"], errors='coerce')
assert (df_clean["fee"] >= 0).all()

df_clean.to_csv("cleaned_transactions_final.csv", index=False)


In [33]:
print("========== TEAM HANDOVER VALIDATION REPORT ==========")
print(f"1. Total Rows Preserved: {len(df_clean)}")
print(f"2. Total Columns: {len(df_clean.columns)}")
print(f"3. Duplicate Rows: {df_clean.drop(columns=['input_amounts_parsed', 'output_amounts_parsed', 'input_addresses_parsed', 'output_addresses_parsed']).duplicated().sum()}")
print(f"4. Duplicate TXIDs: {df_clean['txid'].duplicated().sum()}")
print(f"5. Negative Fees Found: {(df_clean['fee'] < 0).sum()}")
print("\nRemaining Missing Values Per Column:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print("=====================================================")


========== TEAM HANDOVER VALIDATION REPORT ==========
1. Total Rows Preserved: 50000
2. Total Columns: 22
3. Duplicate Rows: 0
4. Duplicate TXIDs: 0
5. Negative Fees Found: 0

Remaining Missing Values Per Column:
src_ip                    1039
src_port                  1039
dst_ip                    1039
input_addresses            506
input_amounts              506
input_amounts_parsed       506
input_addresses_parsed     506
dtype: int64


In [ ]:
df

,timestamp,src_ip,src_port,dst_ip,dst_port,geo_country,asn,txid,input_addresses,output_addresses,input_amounts,output_amounts,fee,script_type,is_anomaly,pattern_type
0,1785524364,191.38.23.68,1924.0,141.12.143.177,8333,US,AS15169,0x8e2ea753485c4d6280c6162751e4128f43d15bceab12...,bc1qd5ceb396aa905cf7413b8ca35718,bc1q357c85c3c26dbc5d499b784c873a;bc1pd6054fe66...,1.1649,0.5821;0.5821,0.00077,p2sh,0,normal
1,1785524953,210.68.57.187,41855.0,48.238.118.132,8333,US,AS15169,0x7855e43fcb8634e5a45526001edd480c32fd0fba778b...,bc1q163eb005199626c8144b4bc12f40,1878492084022132c52fc352bf2e4;bc1qfb13c765e04e...,0.157,0.0784;0.0784,0.00025,v0_p2wpkh,0,normal
2,1785525568,201.212.50.173,31495.0,85.138.195.222,8333,DE,AS3320,0x5e6d0069dc8b60c0943a146b544d3e92e65787438ae3...,16a0510ca1a7f26978b421a99d43c;bc1q45947e2b9ebf...,1f6cc470ba322fca5de8f4d35360f;bc1qc06cff562df9...,0.678;0.678,0.6775;0.6775,0.00098,p2pkh,0,normal
3,1785526850,206.137.211.130,13241.0,212.121.13.166,8333,JP,AS2516,0x01df5f1b25cc0e9af79c33d746a99fcb45a0e2e8174f...,bc1q93722a50b2fe81788ac3035f86f0;1222043552f51...,1bd82a121b93ad54fcbba60309a67,0.5353;0.5353,1.0702,0.00054,p2sh,0,normal
4,1785526859,191.56.238.116,443.0,212.34.74.15,8333,SC,AS9009,0xf72309ed760e7ecfb2e6106589fa4e2a51d817b68546...,bc1qfc8cc112c8dfbfa32a8d517852c0,16785556c2747f7a7881e280998d5;176931d14707a92c...,51.5352,2.1734;2.1734;2.1734;2.1734;2.1734;2.1734;2.17...,1.54610,p2pkh,1,ransom_fanout
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,1830575345,149.167.206.172,1727.0,203.229.188.194,8333,JP,AS2516,0x8353f4445f1fb4382a2d687372d1c7109e4824b8bf14...,1a1046f19911509f493a6fed026f6;306da954363ae135...,104d6606438f46d9aaaef409729a6;31efcbcfd227a9b6...,0.4341;0.4341,0.4339;0.4339,0.00044,p2pkh,0,normal
49996,1830576322,NaN,NaN,NaN,8333,NaN,NaN,0x55b1a4b3d5fe1f7f0ccffef1015cc094fc60d71dd33f...,bc1q9ca6fd49deac66dd2ff3f1e256e6,bc1p7e964c81ae367e820e760ab04f5bf72b;bc1q4336d...,0.5647,0.282;0.282,0.00061,v1_p2tr,0,normal
49997,1830577572,193.199.63.136,38618.0,207.11.189.59,8333,US,AS15169,0xd1b513ea3a41d067d2c717e88d4b66a249da243b2c30...,1bf315632f584a38fe9f65edc8000,bc1qfcb0038c5fab2e2c0b640c1f553d,2.097,2.0965,0.00053,v0_p2wpkh,0,normal
49998,1830578864,221.108.180.212,8732.0,208.43.4.140,8333,DE,AS3320,0xbaaf9f9cff618d3503ecc0abcada9604fba01e3d4a01...,1872f2cbf06bfa1c63181b6a95669;bc1q63c29b01d234...,100d1c15f4720086b57bbd628a176;39d6a4c140b5590e...,0.7385;0.7385,0.7382;0.7382,0.00060,p2pkh,0,normal
